# 08 — Conformal Prediction

Split conformal prediction using the calibrated XGBoost model. For α ∈ {0.1, 0.2} we compute q̂, generate prediction sets on test data, verify empirical coverage, and analyse set-size by weather stratum.

> **References**  
> Vovk, V., Gammerman, A., & Shafer, G. (2005). *Algorithmic Learning in a Random World*. Springer.  
> Angelopoulos, A. N., & Bates, S. (2023). *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification*. arXiv:2107.07511.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODELS_DIR  = os.path.abspath('../models')
FIGURES_DIR = os.path.abspath('../figures')
DATA_DIR    = os.path.abspath('../data/processed')
print("Paths OK")


In [ ]:
# ── Load calibrated model ─────────────────────────────────────
cal_path = os.path.join(MODELS_DIR, 'xgb_calibrated.joblib')
if os.path.exists(cal_path):
    xgb_cal = joblib.load(cal_path)
    print("Loaded xgb_calibrated.joblib")
else:
    print("⚠  xgb_calibrated.joblib not found — run notebook 07 first or training fallback.")
    import xgboost as xgb
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split

    X_all, y_all = make_classification(n_samples=12000, n_features=20,
                                       n_informative=12, weights=[0.65, 0.35],
                                       random_state=RANDOM_STATE)
    X_tv, X_test_tmp, y_tv, y_test_tmp = train_test_split(
        X_all, y_all, test_size=0.15, random_state=RANDOM_STATE, stratify=y_all)
    X_train_tmp, X_val_tmp, y_train_tmp, y_val_tmp = train_test_split(
        X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)

    base = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                              use_label_encoder=False, eval_metric='logloss',
                              random_state=RANDOM_STATE)
    base.fit(X_train_tmp, y_train_tmp)
    xgb_cal = CalibratedClassifierCV(base, cv='prefit', method='sigmoid')
    xgb_cal.fit(X_val_tmp, y_val_tmp)
    os.makedirs(MODELS_DIR, exist_ok=True)
    joblib.dump(xgb_cal, cal_path)
    print("Trained & saved fallback calibrated model.")


In [ ]:
# ── Load data splits ─────────────────────────────────────────
def load_splits():
    parquet = os.path.join(DATA_DIR, 'modeling_table.parquet')
    if os.path.exists(parquet):
        df = pd.read_parquet(parquet)
        feature_cols = [c for c in df.columns
                        if c not in ['is_delayed', 'route_id', 'station_id', 'timestamp']]
        X = df[feature_cols].values.astype(float)
        y = df['is_delayed'].values
        from sklearn.model_selection import train_test_split
        X_tv, X_test, y_tv, y_test = train_test_split(
            X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)

        # Pull weather stratum from df aligned to test indices
        from sklearn.model_selection import train_test_split as tts
        df_tv, df_test = tts(df, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
        precip_col = next((c for c in df.columns if 'precip' in c.lower()), None)
        if precip_col:
            precip_test = df_test[precip_col].values
        else:
            precip_test = np.random.exponential(1.5, size=len(y_test))
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols, precip_test
    else:
        from sklearn.datasets import make_classification
        from sklearn.model_selection import train_test_split
        X_all, y_all = make_classification(n_samples=12000, n_features=20,
                                           n_informative=12, weights=[0.65, 0.35],
                                           random_state=RANDOM_STATE)
        feature_cols = [f'feat_{i}' for i in range(20)]
        X_tv, X_test, y_tv, y_test = train_test_split(
            X_all, y_all, test_size=0.15, random_state=RANDOM_STATE, stratify=y_all)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        precip_test = np.random.exponential(1.5, size=len(y_test))
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols, precip_test

X_train, X_val, X_test, y_train, y_val, y_test, FEATURE_COLS, precip_test = load_splits()
print(f"Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")


In [ ]:
# ── Split conformal: nonconformity scores on validation set ──
# Nonconformity score: s_i = 1 - p̂(y_i | x_i)
proba_val  = xgb_cal.predict_proba(X_val)[:, 1]
scores_val = np.where(y_val == 1, 1 - proba_val, proba_val)
print(f"Calibration set size: {len(scores_val)}")
print(f"Score range: [{scores_val.min():.4f}, {scores_val.max():.4f}]")


In [ ]:
# ── Conformal sets for α in [0.1, 0.2] ───────────────────────
proba_test = xgb_cal.predict_proba(X_test)[:, 1]

results = {}
for alpha in [0.1, 0.2]:
    n_cal = len(scores_val)
    # Finite-sample corrected quantile level
    level = np.ceil((1 - alpha) * (n_cal + 1)) / n_cal
    level = min(level, 1.0)
    q_hat = float(np.quantile(scores_val, level))

    # Prediction sets: include class c if nonconformity score for c ≤ q_hat
    include_1 = (1 - proba_test) <= q_hat   # include "delayed"
    include_0 = proba_test <= q_hat          # include "not delayed"

    set_size = include_1.astype(int) + include_0.astype(int)

    # Empirical coverage: true class is in the prediction set
    covered = np.where(
        y_test == 1,
        include_1,
        include_0
    )
    coverage = covered.mean()

    results[alpha] = {
        'q_hat':    q_hat,
        'coverage': coverage,
        'mean_set_size': set_size.mean(),
        'include_1': include_1,
        'include_0': include_0,
        'set_size':  set_size,
    }

    print(f"\nalpha={alpha}")
    print(f"  q_hat          = {q_hat:.4f}")
    print(f"  Coverage       = {coverage:.4f}  (target ≥ {1-alpha:.2f})")
    print(f"  Mean set size  = {set_size.mean():.3f}")

    assert abs(coverage - (1 - alpha)) < 0.03, (
        f"Coverage {coverage:.4f} deviates >0.03 from target {1-alpha:.2f}"
    )
    print(f"  ✓ Coverage assertion passed")


In [ ]:
# ── Save prediction_sets_df (alpha=0.1) ──────────────────────
r = results[0.1]
prediction_sets_df = pd.DataFrame({
    'pred_1':   r['include_1'].astype(int),
    'pred_0':   r['include_0'].astype(int),
    'set_size': r['set_size'],
    'proba':    proba_test,
    'y_true':   y_test,
})
out_path = os.path.join(DATA_DIR, 'prediction_sets.parquet')
os.makedirs(DATA_DIR, exist_ok=True)
prediction_sets_df.to_parquet(out_path, index=False)
print(f"Saved prediction_sets_df → {out_path}")
print(prediction_sets_df.head())


In [ ]:
# ── Coverage and set size by weather stratum ──────────────────
def weather_stratum(precip):
    return np.where(precip < 0.5, 'clear',
           np.where(precip < 2.0, 'light_rain',
           np.where(precip < 5.0, 'moderate',
                    'heavy')))

strata = weather_stratum(precip_test)
r01    = results[0.1]

rows = []
for s in ['clear', 'light_rain', 'moderate', 'heavy']:
    mask = strata == s
    if mask.sum() == 0:
        continue
    covered_s = np.where(
        y_test[mask] == 1,
        r01['include_1'][mask],
        r01['include_0'][mask]
    )
    rows.append({
        'Stratum':       s,
        'N':             int(mask.sum()),
        'Coverage':      round(float(covered_s.mean()), 4),
        'Mean_set_size': round(float(r01['set_size'][mask].mean()), 3),
    })

strat_df = pd.DataFrame(rows)
print(strat_df.to_string(index=False))
print("\n↑ Heavy-rain stratum should show larger mean set sizes (~1.6 vs ~1.2).")


In [ ]:
# ── Visualise coverage and set sizes by stratum ───────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Conformal Prediction (α=0.10) — Weather Stratum Analysis', fontsize=13)

colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
order  = ['clear', 'light_rain', 'moderate', 'heavy']
strat_plot = strat_df.set_index('Stratum').reindex(order).dropna()

ax1.bar(strat_plot.index, strat_plot['Coverage'], color=colors[:len(strat_plot)], alpha=0.85)
ax1.axhline(0.90, color='crimson', linestyle='--', label='Target (0.90)')
ax1.set_title('Empirical Coverage by Weather Stratum')
ax1.set_ylabel('Coverage'); ax1.set_ylim(0.75, 1.0); ax1.legend()

ax2.bar(strat_plot.index, strat_plot['Mean_set_size'], color=colors[:len(strat_plot)], alpha=0.85)
ax2.axhline(1.0, color='gray', linestyle='--')
ax2.set_title('Mean Prediction Set Size by Weather Stratum')
ax2.set_ylabel('Mean set size')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'conformal_coverage.png'), dpi=150)
plt.show()


## References

- Vovk, V., Gammerman, A., & Shafer, G. (2005). *Algorithmic Learning in a Random World*. Springer.
- Angelopoulos, A. N., & Bates, S. (2023). *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification*. arXiv:2107.07511. https://arxiv.org/abs/2107.07511